# Graded Guided Exercise: Python, `unittest`, `pandas`, and `pytest`
## Please Complete "Inadequate Product Information" Before Moving Into This Section
Your previous project needs to be complete in order for this project to work. We will be adding new code on top of code that should already be implemented and tested successfully in the last project.

## Project Setup
Your project should be initiated and the previous assignment should be fully complete before beginning this assignment. The unit testing performed in this project will build on what was already developed in the previous project.

## Context
We'll be adding new keys into our `products` tables: `price` and `barcode`

We will be pulling these pieces of data from `faker` as we've pulled our other pieces of data. `price` will specifically reference [pricetag()](https://faker.readthedocs.io/en/stable/providers/faker.providers.currency.html), a string data type, while `barcode` will reference [ean](https://faker.readthedocs.io/en/master/providers/faker.providers.barcode.html) with a fixed length of 13 (barcodes either follow 8 digit or 13 digit patterns).


## Open Your Product-Info-Quality Project
1. Open your product-info-quality folder in VSCode

## Add New Columns to Our Tables
To do so, we need to update our `products` model in `models.py` with two new columns called `price` and `barcode`.

2. Inside `models.py`, add two new line to the `Product` class below `information_score`:

In [1]:
    barcode = Column(Numeric, nullable=False, default=0)
    price = Column(String(255), nullable=False, default=0)

NameError: name 'Column' is not defined

3. Since `barcode` is using a `Numeric` data type, we will also need to import that at the top of our `models.py` file
    1. Add `Numeric` to the imports we're pulling from `sqlalchemy`:

In [ ]:
from sqlalchemy import Column, Numeric, ForeignKey, Integer, String, Boolean, DateTime, Text

### Seed the New Data to Our `seed.py` file
We will need to set up our helper functions to ensure that when we seed new data, it's pulling from the correct API (`faker`) and that we're providing the correct parameters in the process, such as `ean` needing a distinction between an 8-digit barcode or a 13-digit barcode.

4. Inside `seed.py`, within the `seed_electronics` function, below `attributes`, add the following code:

In [ ]:
        barcode = fake.ean(length=13)
        price=fake.pricetag()

**Be mindful of the indentations. This code block should match the indentation of the `attributes` line.**

With this, we are establishing where the data is to be pulled from and it will do so at the same time as the other columns.

Right below this like will be a new instance of the `Product` class, `product=Product`, in which we specify which column our seeded data belongs to. We need to add `barcode` and `price` to this instance.

5. Below `description`, add `barcode` and `price` with the following lines:

In [ ]:
            barcode = barcode,
            price=price

**This is one indentation further than the previous line `product=Product`**

We will repeat these steps inside of `seed_grocery`

6. Inside `seed.py`, within the `seed_grocery` function, below `base_description`, add the following code:

In [ ]:
        barcode = fake.ean(length=13)
        price = fake.pricetag()

7. Inside the next new instance of `Product`, below `description`, add `barcode` and `price` with the following lines:

In [ ]:
            barcode = barcode,
            price=price

## Gentle Reminder
We do need an active virtual environment to run this project. Here are the commands for each respective operating system in order to activate the virtual environment:

**macOS / Linux:**
    
    source .venv/bin/activate

**Windows (with Powershell terminal):**

    .venv\Scripts\Activate.ps1

**Windows (CLI “Command Prompt):**

    .venv\Scripts\activate.bat

If you are getting errors about modules such as `faker`, `sqlalchemy`, etc, missing, make sure they are all installed with the following terminal command:

`pip install sqlalchemy streamlit pandas faker pytest`

### Missing Columns!!
If you have already activated the virtual environment and installed the modules, your next error is very likely a **SQL** error! Let us not forget we  are in fact working with a SQL database and it requires **structured** data. We can't just go injecting new data as we like without informing our database that the changes are coming. We need to communicate with our SQL database that we are making changes to our models through SQL queries commands.

**Remember**: Schemas are one of the first things we design when using data. They determine every step afterwards and force us as developers to be mindful of how we plan to move forward.

## Add `barcode` and `price` Columns to the `products` Table
We'll move out of `seed.py` and into our `product-info-quality.session.sql` to run our SQL queries

8. Ensure your database is connected in SQLTools
9. Inside `product-info-quality.session.sql` add the following code to first add `barcode`:

In [ ]:
ALTER TABLE products
ADD barcode INT

10. Click `Run on active connection` to push the changes to your `products` table
11. Replace `ADD barcode INT` with `ADD price VARCHAR(255)` to add `price`
12. Once more, click `Run on active connection` to push the changes to your `products` table
13. If you received no errors, you can check that your `products` table is updated if you click into the SQLTools extension, select your project, select `Tables`, right-click `products` and click `Refresh` and you should see both `barcode` and `price` in your columns
14. Now you can run `python seed.py` in the terminal and it should seed data into our tables
15. If you return the to SQLTools extension, right-click the `products` table and select `Show Table Records` you'll see our tables have two new columns (`price` and `barcode`) as well as data fed into those columns


# Testing

We are going to write automated tests using `pytest` that run **against your real seeded database**. These tests will verify that the data was inserted correctly and that the database is in the shape we expect.

This is called **post-seed testing** — we are not testing with fake or temporary data, we are testing the actual state of `test.db` after `seed.py` has been run.

## Why Does This Matter?
In real software development, tests give you confidence that your code is working as expected. If you ever make a change to `seed.py` or `models.py` and accidentally break something, your tests will catch it immediately rather than you discovering the problem later in a harder-to-debug situation.

## Setup

16. Click the **New File** icon in the Explorer sidebar
17. Name it `test_database.py`
    - Make sure it is **not** inside your `.venv` folder
    - It should sit in the root of your project alongside `models.py`, `seed.py`, etc.
18. Make sure the file is empty before continuing

Just like every other Python file in this project, we need to import the tools we'll be using. We need:
- `pytest` — the testing framework itself
- `create_engine` and `sessionmaker` from SQLAlchemy — to connect to the database
- `DBModelBase` and `Product` from our own `models.py` — so we can query the `products` table

19. At the very top of `test_database.py`, add the following imports:

In [ ]:
import pytest
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from models import DBModelBase, Product

Next we are going to create a fixture. A **fixture** in pytest is a reusable setup function that runs before your tests. Think of it as the "before tests" function — it prepares everything a test needs, then cleans up afterward.

We are using `scope="module"` which means the fixture runs **once for the entire file**, rather than once per test. This is more efficient since we are just reading from an existing database and do not need to reset it between tests.

Notice the fixture is named `seeded_session` — this distinguishes it from any other session fixture you may have that connects to an in-memory database. The name makes it clear this fixture is specifically for working with real, seeded data.

Everything **before** `yield` is the setup. Everything **after** `yield` is the teardown (cleanup). The test itself runs at the `yield` line.

20. Below your imports, add the following fixture:

In [ ]:
@pytest.fixture(scope="module")
def seeded_session():
    engine = create_engine("sqlite:///test.db")
    SessionLocal = sessionmaker(bind=engine)
    db = SessionLocal()
    yield db
    db.close()

## Writing Test 1 — Products Were Seeded

Our first test is the most fundamental one: **did `seed.py` actually insert any data?**

We know from `seed.py` that `main()` seeds exactly 25 products — 10 electronics and 15 grocery. So we query the total count and assert it equals 25.

A few things to notice about the structure:
- The function name starts with `test_` — this is **required** for pytest to recognize it as a test
- It accepts `seeded_session` as a parameter — pytest automatically knows to run our fixture and pass the result in
- `assert` is how we check if something is true. If the condition is false, the test fails and pytest shows us the message after the comma

21. Below the fixture, add the following test:

In [ ]:
def test_products_were_seeded(seeded_session):
    """Check that the database has 25 products after seeding."""
    count = seeded_session.query(Product).count()
    assert count == 25, f"Expected 25 products, but found {count}"

## Writing Test 2 — Both Categories Exist

This test goes one level deeper. Rather than just checking the total count, we verify that the **correct breakdown** of categories was seeded — 10 electronics and 15 grocery.

This is useful because a total count of 25 could still pass even if, for example, all 25 products were accidentally inserted as `electronics`. This test catches that scenario.

Notice we are using `.filter_by(category=...)` — this is SQLAlchemy's way of adding a `WHERE` clause to a query, equivalent to `SELECT * FROM products WHERE category = 'electronics'` in SQL.

22. Below  the `test_products_were_seeded` test, add the following test:

In [ ]:
def test_both_categories_exist(seeded_session):
    """Check that both electronics and grocery products exist in the database."""
    electronics = seeded_session.query(Product).filter_by(category="electronics").count()
    grocery = seeded_session.query(Product).filter_by(category="grocery").count()
    assert electronics == 10, f"Expected 10 electronics, but found {electronics}"
    assert grocery == 15, f"Expected 15 grocery products, but found {grocery}"


## Writing Test 3 — All Products Have a Barcode and Price

This test specifically verifies the **new columns added in Week 6**. We want to make sure that every single product in the database has both a `barcode` and a `price` value — no nulls.

We use `.filter()` with an `|` (OR) condition to find any products where **either** field is missing. If the count of those products is 0, we know every row was populated correctly.

This kind of test is called a **data integrity test** — it checks the quality and completeness of your data rather than just whether the code ran.

23. Below Test 2, add the following test:

In [ ]:
def test_all_products_have_barcode_and_price(seeded_session):
    """Check that every product has a non-null barcode and price."""
    missing = seeded_session.query(Product).filter(
        (Product.barcode == None) | (Product.price == None)
    ).count()
    assert missing == 0, f"{missing} products are missing a barcode or price"


## Run the Tests

Before running the tests, make sure:
- Your virtual environment is active
- You have already run `python seed.py` so `test.db` has data in it
- Make sure your code is properly indented

24. In your terminal, run the following command from the root of your project:

```bash
pytest test_database.py -v
```

The `-v` flag stands for **verbose** — it prints each test name individually so you can see exactly which passed or failed.

A successful run should look like this:

```
test_database.py::test_products_were_seeded                  PASSED
test_database.py::test_both_categories_exist                 PASSED
test_database.py::test_all_products_have_barcode_and_price   PASSED

3 passed in 0.45s
```


## What If a Test Fails?

If a test fails, pytest will tell you exactly which `assert` failed and what the actual values were. For example, if you forgot to run `seed.py` first, you might see:

```
AssertionError: Expected 25 products, but found 0
```

That message is the string we wrote after the comma in our `assert` statement — which is why writing clear assert messages is good practice.

Common things to check if tests fail:
- Did you run `python seed.py` before running the tests?
- Is your virtual environment active?
- Did you run the `ALTER TABLE` SQL queries in SQLTools to add `barcode` and `price` columns?